# Unit Commitment

*Adapted from: [PyPSA Examples &ndash; Unit Commitment](https://docs.pypsa.org/latest/examples/unit-commitment/)*

In [1]:
import datetime as dt

import pandas as pd

import typsa
from typsa.components import (
    Bus,
    CommittableGenerator,
    ExtendableGenerator,
    Generator,
    Load,
)
from typsa.time_variation import IntegerSnapshots, RangedSeries

In [2]:
def create_network(
    coal_plant: Generator | ExtendableGenerator,
    gas_plant: Generator | ExtendableGenerator,
    load: list[int],
) -> typsa.Network[IntegerSnapshots]:
    network = typsa.Network(
        IntegerSnapshots(range(len(load)), spacing=dt.timedelta(hours=1))
    )

    network.add_components(bus := Bus(name="bus"))
    network.add_components(coal_plant, gas_plant)
    network.add_components(
        Load(name="load", bus=bus.name, p_set=RangedSeries(pd.Series(load)))
    )
    return network


def report_results(
    optimized_network: typsa.network.OptimizedNetwork[IntegerSnapshots],
) -> None:
    dynamic_results = optimized_network.dynamic_results
    print(
        pd.concat(
            {
                "Generator dispatched power": dynamic_results.of_all_generators.p,
                "Committable generator status": dynamic_results.of_committable_generators.status.astype(
                    bool
                ),
            },
            axis="columns",
        )
    )

## Minimum Part Load

In [3]:
optimized_network, _ = create_network(
    CommittableGenerator(
        name="coal",
        bus="bus",
        p_min_pu=0.3,
        marginal_cost=20,
        p_nom=10_000,
    ),
    CommittableGenerator(
        name="gas",
        bus="bus",
        p_min_pu=0.1,
        marginal_cost=70,
        p_nom=1_000,
    ),
    load=[4_000, 6_000, 5_000, 800],
).optimize()

Index(['bus'], dtype='str', name='name')
INFO:linopy.model: Solve problem using Highs solver
INFO:linopy.io: Writing time: 0.05s
INFO:linopy.constants: Optimization successful: 
Status: ok
Termination condition: optimal
Solution: 32 primals, 36 duals
Objective: 3.56e+05
Solver model: available
Solver message: Optimal

INFO:pypsa.optimization.optimize:The shadow-prices of the constraints Generator-com-p-lower, Generator-com-p-upper, Generator-com-transition-start-up, Generator-com-transition-shut-down, Generator-com-status-min_up_time_must_stay_up were not assigned to the network.


Running HiGHS 1.12.0 (git hash: 755a8e0): Copyright (c) 2025 HiGHS under MIT licence terms
MIP linopy-problem-drek2vsy has 36 rows; 32 cols; 84 nonzeros; 24 integer variables (24 binary)
Coefficient ranges:
  Matrix  [1e+00, 1e+04]
  Cost    [2e+01, 7e+01]
  Bound   [1e+00, 1e+00]
  RHS     [1e+00, 6e+03]
Presolving model
30 rows, 12 cols, 58 nonzeros  0s
0 rows, 0 cols, 0 nonzeros  0s
Presolve reductions: rows 0(-36); columns 0(-32); nonzeros 0(-84) - Reduced to empty
Presolve: Optimal

Src: B => Branching; C => Central rounding; F => Feasibility pump; H => Heuristic;
     I => Shifting; J => Feasibility jump; L => Sub-MIP; P => Empty MIP; R => Randomized rounding;
     S => Solve LP; T => Evaluate node; U => Unbounded; X => User solution; Y => HiGHS solution;
     Z => ZI Round; l => Trivial lower; p => Trivial point; u => Trivial upper; z => Trivial zero

        Nodes      |    B&B Tree     |            Objective Bounds              |  Dynamic Constraints |       Work      
Src  Pr

In [4]:
report_results(optimized_network)

         Generator dispatched power        Committable generator status       
name                           coal    gas                         coal    gas
snapshot                                                                      
0                            4000.0    0.0                         True  False
1                            6000.0    0.0                         True  False
2                            5000.0    0.0                         True  False
3                               0.0  800.0                        False   True


## Minimum Up Time

In [5]:
optimized_network, _ = create_network(
    CommittableGenerator(
        name="coal",
        bus="bus",
        p_min_pu=0.3,
        marginal_cost=20,
        p_nom=10_000,
    ),
    CommittableGenerator(
        name="gas",
        bus="bus",
        p_min_pu=0.1,
        marginal_cost=70,
        p_nom=1_000,
        up_time_before=0,
        min_up_time=3,
    ),
    load=[4_000, 800, 5_000, 3_000],
).optimize()

Index(['bus'], dtype='str', name='name')
INFO:linopy.model: Solve problem using Highs solver
INFO:linopy.io: Writing time: 0.02s


Running HiGHS 1.12.0 (git hash: 755a8e0): Copyright (c) 2025 HiGHS under MIT licence terms


INFO:linopy.constants: Optimization successful: 
Status: ok
Termination condition: optimal
Solution: 32 primals, 39 duals
Objective: 3.06e+05
Solver model: available
Solver message: Optimal

INFO:pypsa.optimization.optimize:The shadow-prices of the constraints Generator-com-p-lower, Generator-com-p-upper, Generator-com-transition-start-up, Generator-com-transition-shut-down, Generator-com-up-time, Generator-com-status-min_up_time_must_stay_up were not assigned to the network.


MIP linopy-problem-lh3xnbnh has 39 rows; 32 cols; 95 nonzeros; 24 integer variables (24 binary)
Coefficient ranges:
  Matrix  [1e+00, 1e+04]
  Cost    [2e+01, 7e+01]
  Bound   [1e+00, 1e+00]
  RHS     [1e+00, 5e+03]
Presolving model
33 rows, 15 cols, 71 nonzeros  0s
0 rows, 0 cols, 0 nonzeros  0s
Presolve reductions: rows 0(-39); columns 0(-32); nonzeros 0(-95) - Reduced to empty
Presolve: Optimal

Src: B => Branching; C => Central rounding; F => Feasibility pump; H => Heuristic;
     I => Shifting; J => Feasibility jump; L => Sub-MIP; P => Empty MIP; R => Randomized rounding;
     S => Solve LP; T => Evaluate node; U => Unbounded; X => User solution; Y => HiGHS solution;
     Z => ZI Round; l => Trivial lower; p => Trivial point; u => Trivial upper; z => Trivial zero

        Nodes      |    B&B Tree     |            Objective Bounds              |  Dynamic Constraints |       Work      
Src  Proc. InQueue |  Leaves   Expl. | BestBound       BestSol              Gap |   Cuts   InLp Co

In [6]:
report_results(optimized_network)

         Generator dispatched power        Committable generator status       
name                           coal    gas                         coal    gas
snapshot                                                                      
0                            3900.0  100.0                         True   True
1                               0.0  800.0                        False   True
2                            4900.0  100.0                         True   True
3                            3000.0   -0.0                         True  False


## Minimum Down Time

In [7]:
optimized_network, _ = create_network(
    CommittableGenerator(
        name="coal",
        bus="bus",
        p_min_pu=0.3,
        marginal_cost=20,
        p_nom=10_000,
        min_down_time=2,
        down_time_before=1,
    ),
    CommittableGenerator(
        name="gas",
        bus="bus",
        p_min_pu=0.1,
        marginal_cost=70,
        p_nom=4_000,
    ),
    load=[3_000, 800, 3_000, 8_000],
).optimize()

Index(['bus'], dtype='str', name='name')
INFO:linopy.model: Solve problem using Highs solver
INFO:linopy.io: Writing time: 0.02s
INFO:linopy.constants: Optimization successful: 
Status: ok
Termination condition: optimal
Solution: 32 primals, 40 duals
Objective: 4.86e+05
Solver model: available
Solver message: Optimal

INFO:pypsa.optimization.optimize:The shadow-prices of the constraints Generator-com-p-lower, Generator-com-p-upper, Generator-com-transition-start-up, Generator-com-transition-shut-down, Generator-com-down-time, Generator-com-status-min_up_time_must_stay_up, Generator-com-status-min_down_time_must_stay_up were not assigned to the network.


Running HiGHS 1.12.0 (git hash: 755a8e0): Copyright (c) 2025 HiGHS under MIT licence terms
MIP linopy-problem-nlybi_ik has 40 rows; 32 cols; 94 nonzeros; 24 integer variables (24 binary)
Coefficient ranges:
  Matrix  [1e+00, 1e+04]
  Cost    [2e+01, 7e+01]
  Bound   [1e+00, 1e+00]
  RHS     [1e+00, 8e+03]
Presolving model
27 rows, 11 cols, 52 nonzeros  0s
3 rows, 3 cols, 6 nonzeros  0s
0 rows, 0 cols, 0 nonzeros  0s
Presolve reductions: rows 0(-40); columns 0(-32); nonzeros 0(-94) - Reduced to empty
Presolve: Optimal

Src: B => Branching; C => Central rounding; F => Feasibility pump; H => Heuristic;
     I => Shifting; J => Feasibility jump; L => Sub-MIP; P => Empty MIP; R => Randomized rounding;
     S => Solve LP; T => Evaluate node; U => Unbounded; X => User solution; Y => HiGHS solution;
     Z => ZI Round; l => Trivial lower; p => Trivial point; u => Trivial upper; z => Trivial zero

        Nodes      |    B&B Tree     |            Objective Bounds              |  Dynamic Constra

In [8]:
report_results(optimized_network)

         Generator dispatched power         Committable generator status  \
name                           coal     gas                         coal   
snapshot                                                                   
0                               0.0  3000.0                        False   
1                               0.0   800.0                        False   
2                            3000.0     0.0                         True   
3                            8000.0     0.0                         True   

                 
name        gas  
snapshot         
0          True  
1          True  
2         False  
3         False  


## Start Up and Shut Down Costs

In [9]:
optimized_network, _ = create_network(
    CommittableGenerator(
        name="coal",
        bus="bus",
        p_min_pu=0.3,
        marginal_cost=20,
        p_nom=10_000,
        start_up_cost=5_000,
        min_down_time=2,
    ),
    CommittableGenerator(
        name="gas",
        bus="bus",
        p_min_pu=0.1,
        marginal_cost=70,
        p_nom=4_000,
        shut_down_cost=25,
    ),
    load=[3_000, 800, 3_000, 8_000],
).optimize()

Index(['bus'], dtype='str', name='name')
INFO:linopy.model: Solve problem using Highs solver
INFO:linopy.io: Writing time: 0.02s
INFO:linopy.constants: Optimization successful: 
Status: ok
Termination condition: optimal
Solution: 32 primals, 39 duals
Objective: 4.91e+05
Solver model: available
Solver message: Optimal



Running HiGHS 1.12.0 (git hash: 755a8e0): Copyright (c) 2025 HiGHS under MIT licence terms
MIP linopy-problem-o6nwjy2b has 39 rows; 32 cols; 93 nonzeros; 24 integer variables (24 binary)
Coefficient ranges:
  Matrix  [1e+00, 1e+04]
  Cost    [2e+01, 5e+03]
  Bound   [1e+00, 1e+00]
  RHS     [1e+00, 8e+03]
Presolving model
33 rows, 23 cols, 78 nonzeros  0s
11 rows, 11 cols, 23 nonzeros  0s
4 rows, 6 cols, 9 nonzeros  0s
3 rows, 5 cols, 7 nonzeros  0s
0 rows, 0 cols, 0 nonzeros  0s
Presolve reductions: rows 0(-39); columns 0(-32); nonzeros 0(-93) - Reduced to empty
Presolve: Optimal

Src: B => Branching; C => Central rounding; F => Feasibility pump; H => Heuristic;
     I => Shifting; J => Feasibility jump; L => Sub-MIP; P => Empty MIP; R => Randomized rounding;
     S => Solve LP; T => Evaluate node; U => Unbounded; X => User solution; Y => HiGHS solution;
     Z => ZI Round; l => Trivial lower; p => Trivial point; u => Trivial upper; z => Trivial zero

        Nodes      |    B&B Tree 

INFO:pypsa.optimization.optimize:The shadow-prices of the constraints Generator-com-p-lower, Generator-com-p-upper, Generator-com-transition-start-up, Generator-com-transition-shut-down, Generator-com-down-time, Generator-com-status-min_up_time_must_stay_up were not assigned to the network.


In [10]:
report_results(optimized_network)

         Generator dispatched power         Committable generator status  \
name                           coal     gas                         coal   
snapshot                                                                   
0                               0.0  3000.0                        False   
1                               0.0   800.0                        False   
2                            3000.0     0.0                         True   
3                            8000.0     0.0                         True   

                 
name        gas  
snapshot         
0          True  
1          True  
2         False  
3         False  


## Ramp Rate Limits

In [11]:
optimized_network, _ = create_network(
    Generator(
        name="coal",
        bus="bus",
        marginal_cost=20,
        ramp_limit_up=0.1,
        ramp_limit_down=0.2,
        p_nom=10_000,
    ),
    Generator(
        name="gas",
        bus="bus",
        marginal_cost=70,
        p_nom=4_000,
    ),
    load=[4_000, 7_000, 7_000, 7_000, 7_000, 3_000],
).optimize()

Index(['bus'], dtype='str', name='name')
INFO:linopy.model: Solve problem using Highs solver
INFO:linopy.io: Writing time: 0.01s
INFO:linopy.constants: Optimization successful: 
Status: ok
Termination condition: optimal
Solution: 12 primals, 40 duals
Objective: 9.50e+05
Solver model: available
Solver message: Optimal

INFO:pypsa.optimization.optimize:The shadow-prices of the constraints Generator-fix-p-lower, Generator-fix-p-upper, Generator-fix-p-ramp_limit_up, Generator-fix-p-ramp_limit_down were not assigned to the network.


Running HiGHS 1.12.0 (git hash: 755a8e0): Copyright (c) 2025 HiGHS under MIT licence terms
LP linopy-problem-iujzozut has 40 rows; 12 cols; 56 nonzeros
Coefficient ranges:
  Matrix  [1e+00, 1e+00]
  Cost    [2e+01, 7e+01]
  Bound   [0e+00, 0e+00]
  RHS     [1e+03, 1e+04]
Presolving model
10 rows, 4 cols, 16 nonzeros  0s
2 rows, 3 cols, 4 nonzeros  0s
2 rows, 3 cols, 4 nonzeros  0s
Presolve reductions: rows 2(-38); columns 3(-9); nonzeros 4(-52) 
Solving the presolved LP
Using EKK dual simplex solver - serial
  Iteration        Objective     Infeasibilities num(sum)
          0     0.0000000000e+00 Ph1: 0(0) 0s
          0     9.5000000000e+05 Pr: 0(0) 0s

Performed postsolve
Solving the original LP from the solution after postsolve

Model name          : linopy-problem-iujzozut
Model status        : Optimal
Objective value     :  9.5000000000e+05
P-D objective error :  0.0000000000e+00
HiGHS run time      :          0.00


In [12]:
report_results(optimized_network)

         Generator dispatched power        
name                           coal     gas
snapshot                                   
0                            4000.0    -0.0
1                            5000.0  2000.0
2                            6000.0  1000.0
3                            7000.0    -0.0
4                            5000.0  2000.0
5                            3000.0    -0.0


In [13]:
optimized_network, _ = create_network(
    extendable_coal_plant := ExtendableGenerator(
        name="coal",
        bus="bus",
        marginal_cost=20,
        ramp_limit_up=0.1,
        ramp_limit_down=0.2,
        capital_cost=100,
    ),
    Generator(
        name="gas",
        bus="bus",
        marginal_cost=70,
        p_nom=4_000,
    ),
    load=[4_000, 7_000, 7_000, 7_000, 7_000, 3_000],
).optimize()

Index(['bus'], dtype='str', name='name')
INFO:linopy.model: Solve problem using Highs solver
INFO:linopy.io: Writing time: 0.02s
INFO:linopy.constants: Optimization successful: 
Status: ok
Termination condition: optimal
Solution: 13 primals, 41 duals
Objective: 1.68e+06
Solver model: available
Solver message: Optimal

INFO:pypsa.optimization.optimize:The shadow-prices of the constraints Generator-fix-p-lower, Generator-fix-p-upper, Generator-ext-p-lower, Generator-ext-p-upper, Generator-ext-p-ramp_limit_up, Generator-ext-p-ramp_limit_down were not assigned to the network.


Running HiGHS 1.12.0 (git hash: 755a8e0): Copyright (c) 2025 HiGHS under MIT licence terms
LP linopy-problem-3b7coll8 has 41 rows; 13 cols; 73 nonzeros
Coefficient ranges:
  Matrix  [1e-01, 1e+00]
  Cost    [2e+01, 1e+02]
  Bound   [0e+00, 0e+00]
  RHS     [3e+03, 7e+03]
Presolving model
16 rows, 7 cols, 42 nonzeros  0s
14 rows, 6 cols, 36 nonzeros  0s
Presolve reductions: rows 14(-27); columns 6(-7); nonzeros 36(-37) 
Solving the presolved LP
Using EKK dual simplex solver - serial
  Iteration        Objective     Infeasibilities num(sum)
          0     0.0000000000e+00 Ph1: 0(0) 0s
          6     1.6750000000e+06 Pr: 0(0) 0s

Performed postsolve
Solving the original LP from the solution after postsolve

Model name          : linopy-problem-3b7coll8
Model status        : Optimal
Simplex   iterations: 6
Objective value     :  1.6750000000e+06
P-D objective error :  0.0000000000e+00
HiGHS run time      :          0.00


In [14]:
optimized_network.static_results.capacity_of(extendable_coal_plant)

PNomOpt(value=5000.0)

In [15]:
report_results(optimized_network)

         Generator dispatched power        
name                           coal     gas
snapshot                                   
0                            4000.0    -0.0
1                            4500.0  2500.0
2                            5000.0  2000.0
3                            5000.0  2000.0
4                            4000.0  3000.0
5                            3000.0    -0.0


In [16]:
optimized_network, _ = create_network(
    CommittableGenerator(
        name="coal",
        bus="bus",
        p_min_pu=0.05,
        ramp_limit_up=0.2,
        ramp_limit_down=0.25,
        p_nom=10_000,
        ramp_limit_start_up=0.1,
        ramp_limit_shut_down=0.15,
    ),
    Generator(
        name="gas",
        bus="bus",
        marginal_cost=70,
        p_nom=10_000,
    ),
    load=[0, 200, 7_000, 7_000, 7_000, 2_000, 0],
).optimize()

Index(['bus'], dtype='str', name='name')
INFO:linopy.model: Solve problem using Highs solver
INFO:linopy.io: Writing time: 0.02s
INFO:linopy.constants: Optimization successful: 
Status: ok
Termination condition: optimal
Solution: 35 primals, 61 duals
Objective: 9.59e+05
Solver model: available
Solver message: Optimal

INFO:pypsa.optimization.optimize:The shadow-prices of the constraints Generator-fix-p-lower, Generator-fix-p-upper, Generator-com-p-lower, Generator-com-p-upper, Generator-com-transition-start-up, Generator-com-transition-shut-down, Generator-com-status-min_up_time_must_stay_up, Generator-com-p-ramp_limit_up, Generator-com-p-ramp_limit_down were not assigned to the network.


Running HiGHS 1.12.0 (git hash: 755a8e0): Copyright (c) 2025 HiGHS under MIT licence terms
MIP linopy-problem-8mul6w87 has 61 rows; 35 cols; 144 nonzeros; 21 integer variables (21 binary)
Coefficient ranges:
  Matrix  [1e+00, 1e+04]
  Cost    [7e+01, 7e+01]
  Bound   [1e+00, 1e+00]
  RHS     [1e+00, 1e+04]
Presolving model
39 rows, 12 cols, 93 nonzeros  0s
14 rows, 8 cols, 40 nonzeros  0s
14 rows, 8 cols, 40 nonzeros  0s
Presolve reductions: rows 14(-47); columns 8(-27); nonzeros 40(-104) 

Solving MIP model with:
   14 rows
   8 cols (4 binary, 0 integer, 0 implied int., 4 continuous, 0 domain fixed)
   40 nonzeros

Src: B => Branching; C => Central rounding; F => Feasibility pump; H => Heuristic;
     I => Shifting; J => Feasibility jump; L => Sub-MIP; P => Empty MIP; R => Randomized rounding;
     S => Solve LP; T => Evaluate node; U => Unbounded; X => User solution; Y => HiGHS solution;
     Z => ZI Round; l => Trivial lower; p => Trivial point; u => Trivial upper; z => Trivial zer

In [17]:
report_results(optimized_network)

         Generator dispatched power         Committable generator status
name                           coal     gas                         coal
snapshot                                                                
0                               0.0     0.0                        False
1                               0.0   200.0                        False
2                            1000.0  6000.0                         True
3                            3000.0  4000.0                         True
4                            4000.0  3000.0                         True
5                            1500.0   500.0                         True
6                               0.0     0.0                        False


## Rolling Horizon

In [18]:
load = [4_000, 5_000, 700, 800, 4_000] * 6
optimized_network, _ = create_network(
    CommittableGenerator(
        name="coal",
        bus="bus",
        p_min_pu=0.3,
        marginal_cost=20,
        ramp_limit_up=1,
        ramp_limit_down=1,
        p_nom=10_000,
        start_up_cost=200,
        shut_down_cost=150,
        min_up_time=3,
        min_down_time=2,
        up_time_before=1,
        ramp_limit_start_up=1,
        ramp_limit_shut_down=1,
    ),
    CommittableGenerator(
        name="gas",
        bus="bus",
        marginal_cost=70,
        p_min_pu=0.1,
        p_nom=1_000,
        start_up_cost=50,
        shut_down_cost=20,
        min_up_time=3,
        up_time_before=2,
    ),
    load=load,
).optimize(horizon=5, overlap=2)

Index(['bus'], dtype='str', name='name')
INFO:linopy.model: Solve problem using Highs solver
INFO:linopy.model:Solver options:
 - horizon: 5
 - overlap: 2
INFO:linopy.io: Writing time: 0.02s
INFO:linopy.constants: Optimization successful: 
Status: ok
Termination condition: optimal
Solution: 240 primals, 418 duals
Objective: 2.23e+06
Solver model: available
Solver message: Optimal

INFO:pypsa.optimization.optimize:The shadow-prices of the constraints Generator-com-p-lower, Generator-com-p-upper, Generator-com-transition-start-up, Generator-com-transition-shut-down, Generator-com-up-time, Generator-com-down-time, Generator-com-status-min_up_time_must_stay_up, Generator-com-p-ramp_limit_up, Generator-com-p-ramp_limit_down were not assigned to the network.


ERROR:   getOptionIndex: Option "horizon" is unknown
ERROR:   getOptionIndex: Option "overlap" is unknown
Running HiGHS 1.12.0 (git hash: 755a8e0): Copyright (c) 2025 HiGHS under MIT licence terms
MIP linopy-problem-c3vf26ie has 418 rows; 240 cols; 1150 nonzeros; 180 integer variables (180 binary)
Coefficient ranges:
  Matrix  [1e+00, 1e+04]
  Cost    [2e+01, 2e+02]
  Bound   [1e+00, 1e+00]
  RHS     [1e+00, 5e+03]
Presolving model
365 rows, 198 cols, 1026 nonzeros  0s
324 rows, 90 cols, 340 nonzeros  0s
53 rows, 44 cols, 136 nonzeros  0s
48 rows, 44 cols, 121 nonzeros  0s
Presolve reductions: rows 48(-370); columns 44(-196); nonzeros 121(-1029) 
Objective function is integral with scale 0.1

Solving MIP model with:
   48 rows
   44 cols (44 binary, 0 integer, 0 implied int., 0 continuous, 0 domain fixed)
   121 nonzeros

Src: B => Branching; C => Central rounding; F => Feasibility pump; H => Heuristic;
     I => Shifting; J => Feasibility jump; L => Sub-MIP; P => Empty MIP; R => Rando

In [19]:
print(
    pd.concat(
        {
            "Generator dispatched power": optimized_network.dynamic_results.of_all_generators.p,
            "Committable generator status": optimized_network.dynamic_results.of_committable_generators.status.astype(
                bool
            ),
        },
        axis="columns",
    )
)

         Generator dispatched power        Committable generator status       
name                           coal    gas                         coal    gas
snapshot                                                                      
0                            3900.0  100.0                         True   True
1                            4900.0  100.0                         True   True
2                               0.0  700.0                        False   True
3                               0.0  800.0                        False   True
4                            4000.0    0.0                         True  False
5                            4000.0    0.0                         True  False
6                            5000.0    0.0                         True  False
7                               0.0  700.0                        False   True
8                               0.0  800.0                        False   True
9                            3900.0  100.0          